# M22 — Open the Neuron and Layer

**Objective:** decompose the network into neurons, activations and layers.

M21 trained a complete estimator without opening internals. M22 asks what
one hidden unit *is*. The useful whole is not a training loop. It is:

`z = w·x + b` then `y = activation(z)`, lifted to a row-batch layer
`Y = activation(X @ W + b)` with `W` shaped `(n_in, n_out)`.

Multi-layer NumPy inference, logits, gradients, and autograd stay closed
(M23-M25).


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a number, a sign, or a shape.

Do not inspect sklearn `coefs_`, do not write `backward`, and do not stack
a full inference network. If a failure can be diagnosed from a two-input
hand calculation or a shape contract, stay there.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M22" / "neuron_layer_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M22.neuron_layer_core import (
    REFERENCE_BIAS,
    REFERENCE_LAYER_BIAS,
    REFERENCE_LAYER_W,
    REFERENCE_LAYER_X,
    REFERENCE_PREACTIVATION,
    REFERENCE_WEIGHTS,
    REFERENCE_X,
    activation_sweep,
    affine_preactivation,
    collapsed_affine,
    compose_two_layers,
    dense_forward,
    dense_forward_with_defect,
    layer_report,
    neuron_trace,
    reference_neuron_relu,
    validate_dense_shapes,
)

print("repository root:", ROOT)
print("reference x:", REFERENCE_X)
print("reference weights:", REFERENCE_WEIGHTS)
print("reference bias:", REFERENCE_BIAS)
print("hand-computed preactivation target:", REFERENCE_PREACTIVATION)
print("layer layout: X (batch, n_in) @ W (n_in, n_out) + b (n_out,)")
print("activation placement: after the affine map")


## M21 boundary: keep the black box closed

M21 froze `hidden_units=64` and `relu` as sklearn knobs. That was a
**capacity and configuration** observation, not a causal story about
neurons.

What this mission **opens:** affine arithmetic, bias, ReLU/sigmoid/tanh
as maps, dense-layer shapes, batches, and two-layer affine composition.

What stays **deferred:**
- M23 — multi-layer NumPy inference, logits, probability normalization
- M24 — backpropagation
- M25 — PyTorch autograd / training loop

Do not reuse M21 test accuracy as evidence that ReLU is “the right”
activation.


## Frozen teaching fixtures

Declare the useful whole **before** the first calculation.

| Fixture | Value |
| --- | --- |
| Neuron input `x` | `(1.0, 2.0)` |
| Neuron weights `w` | `(0.5, -0.25)` |
| Neuron bias `b` | `0.5` |
| Hand-computed `z` | `0.5*1 + (-0.25)*2 + 0.5 = 0.5` |
| Layer `X` | two rows, three features |
| Layer `W` | `(3, 2)` so `n_in=3`, `n_out=2` |
| Layer `b` | `(0.0, 0.5)` |
| Layout | row-batch, same orientation as M16 |

Primary sources: `3b1b-neural-networks` and `fastai-course` in
`data/source_registry.json`. Skip micrograd and PyTorch here.


## Predict before running — one neuron

Timestamp a prediction before `run-neuron`.

For `x=(1, 2)`, `w=(0.5, -0.25)`, `b=0.5`, predict:
- the weighted sum `w·x` (no bias yet)
- the pre-activation `z = w·x + b`
- ReLU(`z`)

If you predict `z = 1.0`, write that down so the observation can falsify it.


In [ ]:
trace = neuron_trace(REFERENCE_X, REFERENCE_WEIGHTS, REFERENCE_BIAS, "relu")
print("x", trace.x)
print("weights", trace.weights)
print("bias", trace.bias)
print("weighted_sum", trace.weighted_sum)
print("preactivation", trace.preactivation)
print("relu output", trace.output)
print("independent affine_preactivation", affine_preactivation(REFERENCE_X, REFERENCE_WEIGHTS, REFERENCE_BIAS))
assert abs(trace.preactivation - REFERENCE_PREACTIVATION) < 1e-12
assert abs(trace.output - 0.5) < 1e-12


### Observe the three pieces separately

The weighted sum here is `0.0`. The bias then translates it to `0.5`.
ReLU leaves a non-negative `z` unchanged. A neuron is this pipeline, not
a biological story and not a training algorithm.


## Predict before running — one feature changes

Timestamp a prediction before `run-feature-change`.

Change **only** `x[0]` from `1.0` to `3.0`. Predict the change in `z`
from the corresponding weight `0.5`. All other inputs and parameters
stay fixed.


In [ ]:
changed = neuron_trace((3.0, 2.0), REFERENCE_WEIGHTS, REFERENCE_BIAS, "relu")
print("new preactivation", changed.preactivation)
print("new output", changed.output)
print("delta z", changed.preactivation - trace.preactivation)
assert abs((changed.preactivation - trace.preactivation) - 1.0) < 1e-12


### One weight explains the delta

The first weight is `0.5` and the feature rose by `2`, so `z` rose by
`1.0`. That is the single-neuron version of "change one named input."
Nothing else moved.


## Predict before running — bias ablation

Timestamp a prediction before `run-bias-ablation`.

Set **only** bias to `0`. Predict:
- `z` equals the weighted sum
- the output translation relative to the reference neuron is `-0.5` when
  using the identity map so ReLU does not clip the comparison


In [ ]:
with_bias = neuron_trace(REFERENCE_X, REFERENCE_WEIGHTS, 0.5, "identity")
no_bias = neuron_trace(REFERENCE_X, REFERENCE_WEIGHTS, 0.0, "identity")
print("z with bias", with_bias.preactivation)
print("z without bias", no_bias.preactivation)
print("translation", with_bias.preactivation - no_bias.preactivation)
assert abs(no_bias.preactivation - no_bias.weighted_sum) < 1e-12
assert abs((with_bias.preactivation - no_bias.preactivation) - 0.5) < 1e-12


### Bias is a translation of z

Zeroing bias does not zero the weights. It removes an offset. That is
enough to explain why “turn the bias off” changes a neuron without
opening a gradient tape.


## Predict before running — activation sweep

Timestamp a prediction before `run-activation-sweep`.

Use the shared pre-activation sequence `[-2, -0.5, 0, 0.5, 2]`. Predict:
- ReLU is `0` on every negative input and copies the positives
- sigmoid at `0` is `0.5` and stays in `(0, 1)`
- tanh at `0` is `0` and stays in `[-1, 1]`

Do not rank the three as universally better or worse.


In [ ]:
zs = (-2.0, -0.5, 0.0, 0.5, 2.0)
swept = activation_sweep(zs)
print("preactivations", zs)
for name, values in swept.items():
    print(name, values)
assert swept["relu"] == (0.0, 0.0, 0.0, 0.5, 2.0)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
grid = [i / 20 for i in range(-60, 61)]
for name in ("relu", "sigmoid", "tanh"):
    ax.plot(grid, activation_sweep(grid, names=(name,))[name], label=name)
ax.axhline(0, color="0.7", linewidth=0.8)
ax.axvline(0, color="0.7", linewidth=0.8)
ax.set(xlabel="pre-activation z", ylabel="activation(z)", title="Same z, three maps")
ax.legend()
fig.tight_layout()
plt.show()


### Activations are observed maps

ReLU is a hinge. Sigmoid and tanh saturate. These plots justify only
statements about this sequence. They do not justify “always use ReLU
because M21 scored 0.95.”


## Predict before running — dense layer shapes

Timestamp a prediction before `run-dense-layer`.

`X` is `(2, 3)`, `W` is `(3, 2)`, `b` is `(2,)`. Predict:
- `Y` is `(2, 2)`
- first row after ReLU is `(0, 0)` because `z = (0, -0.5)`
- second row after ReLU is `(1.0, 1.5)`

Activation happens **after** `X @ W + b`.


In [ ]:
batch_shape = validate_dense_shapes(REFERENCE_LAYER_X, REFERENCE_LAYER_W, REFERENCE_LAYER_BIAS)
print("batch, n_in, n_out", batch_shape)
layer_out = dense_forward(REFERENCE_LAYER_X, REFERENCE_LAYER_W, REFERENCE_LAYER_BIAS, "relu")
print("Y shape", layer_out.shape)
print("layer_report", layer_report(layer_out))
assert layer_out.shape == (2, 2)
assert abs(float(layer_out[0, 0])) < 1e-12
assert abs(float(layer_out[0, 1])) < 1e-12
assert abs(float(layer_out[1, 0]) - 1.0) < 1e-12
assert abs(float(layer_out[1, 1]) - 1.5) < 1e-12


### Annotate every dimension

`X`: batch × features. `W`: features × neurons. `b`: neurons.
`Y`: batch × neurons. One column of `W` is one neuron's weights. This is
the M16 row-batch convention, not a column-vector lecture hiding in a
new name.


## Predict before running — singleton versus batch

Timestamp a prediction before `run-batch`.

Predict that running `dense_forward` on each row alone reproduces the
corresponding batch row, with singleton shape `(1, n_out)`.


In [ ]:
for index, row in enumerate(REFERENCE_LAYER_X):
    single = dense_forward(row, REFERENCE_LAYER_W, REFERENCE_LAYER_BIAS, "relu")
    print("row", index, "singleton", single, "batch row", layer_out[index])
    assert single.shape == (1, 2)
    assert abs(float(single[0, 0]) - float(layer_out[index, 0])) < 1e-12
    assert abs(float(single[0, 1]) - float(layer_out[index, 1])) < 1e-12
print("per-row equivalence holds")


### A batch is stacked singletons

If row `i` of the batch disagrees with the singleton run on that row,
the layout is wrong. M16 already used this row-batch test; M22 reuses
it on a neuron layer.


## Predict before running — linearity collapse

Timestamp a prediction before `run-linearity`.

Compose two affine maps. Predict:
- hidden `identity` matches `collapsed_affine` exactly
- hidden ReLU does **not** match that collapsed map on this fixture

Same `X`, `W1`, `b1`, `W2`, `b2` in both runs. Only the hidden map
changes.


In [ ]:
w2 = ((1.0, 0.0), (0.0, 1.0))
b2 = (0.25, -0.25)
identity_hidden = compose_two_layers(
    REFERENCE_LAYER_X, REFERENCE_LAYER_W, REFERENCE_LAYER_BIAS, w2, b2,
    hidden_activation="identity",
)
weights_eq, bias_eq = collapsed_affine(REFERENCE_LAYER_W, REFERENCE_LAYER_BIAS, w2, b2)
collapsed = dense_forward(REFERENCE_LAYER_X, weights_eq, bias_eq, "identity")
relu_hidden = compose_two_layers(
    REFERENCE_LAYER_X, REFERENCE_LAYER_W, REFERENCE_LAYER_BIAS, w2, b2,
    hidden_activation="relu",
)
print("collapsed W", weights_eq)
print("collapsed b", bias_eq)
print("identity hidden", identity_hidden)
print("collapsed affine", collapsed)
print("relu hidden", relu_hidden)
assert abs(float((identity_hidden - collapsed).max())) < 1e-12
assert abs(float((relu_hidden - collapsed).max())) > 1e-12


### Nonlinearity is what stacking buys

Two affine maps are still one affine map. A hidden ReLU is the first
place this teaching stack can stop being a single linear transform.
That is the representation fact M22 needs. It is not a deeper-is-always
better claim.


## Code reading — validate, affine, then activate

Read `dense_forward` and `validate_dense_shapes` in
`missions/M22/neuron_layer_core.py` (see also `missions/M22/code_reading.md`).
Before the next cell, predict:

1. what error you get if `X` has 2 features and `W` has 3 rows
2. whether a 1-D `x` becomes a batch of one row
3. whether activation can run before `X @ W`

Do not search the file for a backward pass.


In [ ]:
source = inspect.getsource(dense_forward)
markers = ("validate_dense_shapes", "@", "apply_activation")
print("dense_forward markers")
for marker in markers:
    print(f"  {marker!r} present: {marker in source}")
try:
    validate_dense_shapes(((1.0, 2.0),), REFERENCE_LAYER_W, REFERENCE_LAYER_BIAS)
    raise AssertionError("expected a feature/weight mismatch")
except ValueError as exc:
    print("malformed shape:", exc)


## Predict before running — Controlled failure: wrong weight orientation

Timestamp a prediction before `run-failure`.

Square fixture: `x=(1, 2)`, `W=[[1, 3], [0, 2]]`, `b=(0, 0)`, identity map.

Predict:
- correct `X @ W` is `(1, 7)`
- `transposed_weights` still emits finite numbers `(7, 4)`
- this is an orientation defect, not a reason to add a layer or a gradient


In [ ]:
x_sq = ((1.0, 2.0),)
w_sq = ((1.0, 3.0), (0.0, 2.0))
b_sq = (0.0, 0.0)
correct_sq = dense_forward(x_sq, w_sq, b_sq, "identity")
broken_sq = dense_forward_with_defect(x_sq, w_sq, b_sq, "identity", defect="transposed_weights")
print("correct X @ W", correct_sq)
print("defective X @ W.T", broken_sq)
assert abs(float(correct_sq[0, 1]) - 7.0) < 1e-12
assert abs(float(broken_sq[0, 1]) - 4.0) < 1e-12


### Diagnose before repair

The intended contract is `X @ W` with `W` shaped `(n_in, n_out)`. The
defective path multiplied by `W.T`. Both outputs are finite. The
hand-computed `(1, 7)` discriminates them.

Do not repair this by changing the teaching activation or by opening M23.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

Predict that `defect="none"` restores `(1, 7)` exactly, with the same
`X`, `W`, and `b`.


In [ ]:
repaired = dense_forward_with_defect(x_sq, w_sq, b_sq, "identity", defect="none")
print("repaired", repaired)
assert abs(float((repaired - correct_sq).max())) < 1e-12
print("orientation repair restored the hand-computed layer")


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- the hand-computed neuron (`z=0.5`)
- the activation sweep on the shared sequence
- annotated `X`, `W`, `b`, `Y` shapes
- linearity collapse comparison
- orientation-defect diagnosis and smallest repair
- the code-reading trace

See `missions/M22/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M22/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M22/adr_prompt.md` to choose a V05 **teaching**
activation/width policy. Do not claim global optimality.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR.


## M21 → M22 → M23 handoff

M21 trains the whole network. M22 opens the neuron and one dense layer.
M23 may reconstruct a multi-layer NumPy forward pass **only after** these
equations, shapes, activation placement, and hand-computed references
are defended.

M23 must not invent a different layout without stating the change. M24
still owns gradients.


## Mission summary prompt

In your own words, using only numbers from this lab:

1. Why is bias a translation of `z` rather than a feature weight?
2. Why do two affine layers without a nonlinearity collapse?
3. How did the hand-computed `(1, 7)` diagnose `W.T`?
4. What must M23 receive that M21 could not provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert abs(trace.preactivation - 0.5) < 1e-12
assert layer_out.shape == (2, 2)
assert abs(float((identity_hidden - collapsed).max())) < 1e-12
assert abs(float(correct_sq[0, 1]) - 7.0) < 1e-12
print("M22 integrity checks passed")
